# LeetCode 2733 — Neither Minimum nor Maximum bằng Python

Link đề: https://leetcode.com/problems/neither-minimum-nor-maximum/ · Độ khó: **Easy** · Chủ đề: Array, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findNonMinOrMax(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **một** phần tử của mảng **không phải nhỏ nhất, cũng không phải lớn nhất**; nếu không có thì trả `-1`. Đề chấp nhận **mọi** đáp án hợp lệ ("any").
2. Biết vì sao các phần tử **đôi một khác nhau** (distinct) giúp bài đơn giản: min và max là **hai phần tử duy nhất**, nên mảng có ít hơn 3 phần tử thì chắc chắn không có đáp án.
3. Dùng `min`, `max`, so sánh dây chuyền `lo < x < hi`, `sorted`, lát cắt `nums[:3]`, `next(..., default)`.
4. Hiểu thủ thuật **"3 phần tử đầu"**: trung vị của ba phần tử bất kỳ khác nhau luôn không phải min/max của cả mảng ⇒ lời giải **`O(1)`**.
5. Viết 4 cách: thử từng phần tử (brute force), `min`/`max` rồi quét, sắp xếp lấy phần tử thứ hai, và trung vị của 3 phần tử đầu.
6. Biết cách **viết bộ kiểm tra hợp lệ (checker)** khi đề chấp nhận nhiều đáp án, thay vì so sánh bằng `==`.
7. Tránh các lỗi quen thuộc: trả luôn `nums[1]`, quên trường hợp `n < 3`, trả chỉ số thay vì giá trị, tự "chế" số không có trong mảng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` gồm các số **dương, đôi một khác nhau**. Hãy tìm và trả về **một** số trong mảng mà **không phải giá trị nhỏ nhất** cũng **không phải giá trị lớn nhất** của mảng. Nếu **không có** số nào như vậy, trả về `-1`.

Đề cho phép trả về **bất kỳ** số thoả mãn.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[3,2,1,4]` | `2` (hoặc `3`) | min là `1`, max là `4`; cả `2` và `3` đều hợp lệ |
| 2 | `[1,2]` | `-1` | chỉ có min `1` và max `2`, không còn số nào khác |
| 3 | `[2,1,3]` | `2` | min `1`, max `3`; chỉ `2` hợp lệ |

Constraints: `1 <= nums.length <= 100`; `1 <= nums[i] <= 100`; mọi giá trị trong `nums` **khác nhau**.

In [1]:
for nums in ([3, 2, 1, 4], [1, 2], [2, 1, 3]):
    lo, hi = min(nums), max(nums)
    ok = [x for x in nums if x != lo and x != hi]
    print(nums, "| min =", lo, "max =", hi, "| các đáp án hợp lệ:", ok or "không có (trả -1)")

[3, 2, 1, 4] | min = 1 max = 4 | các đáp án hợp lệ: [3, 2]
[1, 2] | min = 1 max = 2 | các đáp án hợp lệ: không có (trả -1)
[2, 1, 3] | min = 1 max = 3 | các đáp án hợp lệ: [2]


## 3. Kiến thức nền

### 3.1. `min`, `max`, so sánh dây chuyền

`min(nums)` / `max(nums)` cho giá trị nhỏ nhất / lớn nhất. Python cho phép **so sánh dây chuyền**: `lo < x < hi` nghĩa là `lo < x and x < hi` (viết gọn, đọc như toán).

In [2]:
nums = [3, 2, 1, 4]
lo, hi = min(nums), max(nums)
print(lo, hi)                                  # 1 4
print(lo < 3 < hi, lo < 1 < hi, lo < 4 < hi)   # True False False
print([x for x in nums if lo < x < hi])        # [3, 2]

1 4
True False False
[3, 2]


### 3.2. Phần tử đôi một khác nhau (distinct) và vì sao quan trọng

Vì giá trị đều khác nhau nên **chỉ có một phần tử bằng min và một phần tử bằng max**. Nếu `n ≥ 3`, ngoài hai phần tử đó còn ít nhất một phần tử nữa — chắc chắn nằm **thật sự giữa** min và max. Nếu `n ≤ 2` thì mọi phần tử là min hoặc max.

Nếu cho phép trùng giá trị (ví dụ `[2, 2, 2]`) thì mọi chuyện khác: có thể có nhiều số bằng min, nhiều số bằng max, nên điều kiện `n ≥ 3` không còn đủ.

In [3]:
for nums in ([5], [5, 7], [5, 7, 6]):
    print(len(nums), nums, "-> có đáp án?", len(nums) >= 3)
print(len(set([3, 2, 1, 4])) == len([3, 2, 1, 4]))      # True: các phần tử khác nhau

1 [5] -> có đáp án? False
2 [5, 7] -> có đáp án? False
3 [5, 7, 6] -> có đáp án? True
True


### 3.3. Lát cắt `nums[:3]` và `sorted`

`nums[:3]` lấy tối đa 3 phần tử đầu (không lỗi nếu mảng ngắn hơn). `sorted(a)` trả list mới đã sắp tăng dần; `sorted(a)[1]` là phần tử **nhỏ thứ hai** (chỉ số 1).

In [4]:
nums = [3, 2, 1, 4]
print(nums[:3], sorted(nums[:3]), sorted(nums[:3])[1])   # [3, 2, 1] [1, 2, 3] 2
print([5][:3], [5, 7][:3])                                # [5] [5, 7]
print(sorted(nums), sorted(nums)[1])                      # [1, 2, 3, 4] 2

[3, 2, 1] [1, 2, 3] 2
[5] [5, 7]
[1, 2, 3, 4] 2


### 3.4. `next(iterator, default)`: lấy phần tử đầu tiên thoả điều kiện

`next(gen, default)` trả phần tử **đầu tiên** của bộ sinh (generator) hoặc `default` nếu không có phần tử nào. Rất hợp để "tìm một phần tử thoả điều kiện, nếu không có thì `-1`".

In [5]:
nums = [3, 2, 1, 4]
lo, hi = min(nums), max(nums)
print(next((x for x in nums if lo < x < hi), -1))       # 3
print(next((x for x in [1, 2] if 1 < x < 2), -1))        # -1

3
-1


### 3.5. Khi đề cho phép "bất kỳ đáp án hợp lệ": viết checker

Nếu nhiều đáp án đều đúng thì **không so sánh** `kết_quả == đáp_án_mẫu`. Thay vào đó viết hàm `is_valid(nums, ans)` kiểm tra **ans có hợp lệ không**. Ở bài này: nếu `n < 3` thì `ans == -1`; ngược lại `ans` phải **thuộc `nums`** và `ans != min(nums)` và `ans != max(nums)`.

In [6]:
def is_valid(nums, ans):
    if len(nums) < 3:
        return ans == -1
    return ans in nums and ans != min(nums) and ans != max(nums)

print(is_valid([3, 2, 1, 4], 2), is_valid([3, 2, 1, 4], 3), is_valid([3, 2, 1, 4], 1))   # True True False
print(is_valid([1, 2], -1), is_valid([1, 2], 1))                                         # True False

True True False
True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Nhìn vào ví dụ `[3,2,1,4]`: min `1`, max `4`. Ngoài hai số này, mọi số còn lại đều "ở giữa" — vậy **bất kỳ số nào ngoài min và max** đều là đáp án.

Câu hỏi cần trả lời: **có ít nhất ba phần tử** hay không? Có ⇒ chắc chắn tồn tại phần tử thứ ba (không phải min cũng không phải max) ⇒ đáp án khác `-1`. Không (≤ 2 phần tử) ⇒ `-1`.

Tính tay:

| nums | n | min | max | có phần tử khác? | đáp án |
|---|---|---|---|---|---|
| `[3,2,1,4]` | 4 | 1 | 4 | `3`, `2` | `3` hoặc `2` |
| `[1,2]` | 2 | 1 | 2 | không | `-1` |
| `[2,1,3]` | 3 | 1 | 3 | `2` | `2` |

In [7]:
def tinh_tay(nums):
    if len(nums) < 3:
        return -1
    lo, hi = min(nums), max(nums)
    for x in nums:
        if x != lo and x != hi:
            return x

print(tinh_tay([3, 2, 1, 4]), tinh_tay([1, 2]), tinh_tay([2, 1, 3]))      # 3 -1 2

3 -1 2


## 5. Cách 1 — Brute Force / baseline: thử từng phần tử

**Ý tưởng:** định nghĩa đúng theo đề: `x` **không phải min** nếu **tồn tại** phần tử nhỏ hơn `x`; **không phải max** nếu **tồn tại** phần tử lớn hơn `x`. Với mỗi `x` trong `nums`: duyệt toàn mảng để kiểm tra hai điều đó; thấy `x` thoả thì trả luôn. Không có thì `-1`.

In [8]:
from typing import List

def find_brute(nums: List[int]) -> int:
    for x in nums:
        has_smaller = False
        has_larger = False
        for y in nums:
            if y < x:
                has_smaller = True
            if y > x:
                has_larger = True
        if has_smaller and has_larger:
            return x
    return -1

print(find_brute([3, 2, 1, 4]), find_brute([1, 2]), find_brute([2, 1, 3]))      # 3 -1 2

3 -1 2


**Độ phức tạp:** vòng ngoài `n`, vòng trong `n` ⇒ Time `O(n²)` (tối đa `100² = 10^4`), Space `O(1)`.

**Nút thắt:** mỗi lần ta **tính lại** "có nhỏ hơn / có lớn hơn" cho từng `x` dù mức nhỏ nhất và lớn nhất của cả mảng là **hai con số cố định**. Tính `min` và `max` **một lần** là xong.

## 6. Ý tưởng thuật toán / bước đột phá

Một số `x` không phải min và không phải max **khi và chỉ khi** `min(nums) < x < max(nums)`. Tính `lo = min(nums)`, `hi = max(nums)` **một lần** (mỗi cái `O(n)`), rồi quét `nums` tìm phần tử đầu tiên có `lo < x < hi` ⇒ **`O(n)`**.

Đột phá thứ hai (tinh tế hơn): ta **không cần biết min và max của cả mảng**! Lấy **3 phần tử bất kỳ** `a, b, c` (đôi một khác nhau). Gọi `m` là **trung vị** của chúng (phần tử ở giữa khi sắp xếp ba số). Khi đó trong mảng có phần tử **nhỏ hơn `m`** (là phần tử nhỏ nhất của ba) và phần tử **lớn hơn `m`** (là phần tử lớn nhất của ba) ⇒ `m` **không thể là min cũng không thể là max** của cả mảng. Vậy chỉ cần xét **3 phần tử đầu** ⇒ `O(1)`.

In [9]:
nums = [3, 2, 1, 4]
a, b, c = nums[:3]
m = a + b + c - min(a, b, c) - max(a, b, c)      # trung vị của 3 số khác nhau
print("trung vị 3 phần tử đầu:", m, "| nhỏ hơn có trong mảng:", any(x < m for x in nums), "| lớn hơn:", any(x > m for x in nums))

trung vị 3 phần tử đầu: 2 | nhỏ hơn có trong mảng: True | lớn hơn: True


## 7. Các cách tiếp theo

Ngoài Cách 1 (brute force), ta xem 3 cách: (2) `min`/`max` rồi quét, (3) sắp xếp lấy phần tử thứ hai, (4) trung vị của **3 phần tử đầu** (`O(1)`).

### Cách 2 — Tính `min`, `max` một lần rồi quét

Giải thích từng đoạn:

- `if len(nums) < 3: return -1` — với mảng khác nhau từng đôi một, ≤ 2 phần tử thì mọi phần tử đều là min hoặc max.
- `lo, hi = min(nums), max(nums)` — hai mốc cố định.
- `for x in nums: if lo < x < hi: return x` — phần tử đầu tiên ở giữa.
- Vì `n ≥ 3` và các giá trị khác nhau, vòng lặp chắc chắn tìm được; `return -1` cuối chỉ để an toàn.

In [10]:
def find_min_max(nums: List[int]) -> int:
    if len(nums) < 3:
        return -1
    lo, hi = min(nums), max(nums)
    for x in nums:
        if lo < x < hi:
            return x
    return -1

print(find_min_max([3, 2, 1, 4]), find_min_max([1, 2]), find_min_max([2, 1, 3]))

3 -1 2


In [11]:
from typing import List

class SolutionMinMax:
    def findNonMinOrMax(self, nums: List[int]) -> int:
        if len(nums) < 3:
            return -1
        lo, hi = min(nums), max(nums)
        for x in nums:
            if lo < x < hi:
                return x
        return -1

print(SolutionMinMax().findNonMinOrMax([3, 2, 1, 4]))

3


**Độ phức tạp:** `min`, `max`, quét: ba lượt `O(n)` ⇒ Time `O(n)`, Space `O(1)`.

### Cách 3 — Sắp xếp, lấy phần tử thứ hai

Sau khi sắp xếp tăng dần, phần tử đầu là min, phần tử cuối là max (giá trị khác nhau), còn **mọi phần tử ở giữa** đều hợp lệ. Lấy `sorted(nums)[1]` (chỉ số `1`) khi `n ≥ 3`. Chú ý: chỉ số `1` luôn **không phải cuối** vì `n ≥ 3`.

In [12]:
def find_sorted(nums: List[int]) -> int:
    if len(nums) < 3:
        return -1
    return sorted(nums)[1]

print(find_sorted([3, 2, 1, 4]), find_sorted([1, 2]), find_sorted([2, 1, 3]))

2 -1 2


In [13]:
from typing import List

class SolutionSorted:
    def findNonMinOrMax(self, nums: List[int]) -> int:
        if len(nums) < 3:
            return -1
        return sorted(nums)[1]

print(SolutionSorted().findNonMinOrMax([3, 2, 1, 4]))

2


**Độ phức tạp:** Time `O(n log n)` do sắp xếp; Space `O(n)` cho bản sao đã sắp (`sorted` không sửa mảng gốc). Đúng nhưng **thừa**: ta đâu cần sắp xếp cả mảng.

### Cách 4 — Trung vị của 3 phần tử đầu (`O(1)`)

Giải thích từng đoạn:

- `if len(nums) < 3: return -1`.
- `a, b, c = nums[0], nums[1], nums[2]` — ba phần tử đầu; đôi một khác nhau theo đề.
- `sorted([a, b, c])[1]` — phần tử ở giữa của ba số (trung vị).
- Chứng minh ở mục 10: trung vị này luôn lớn hơn một phần tử (có trong mảng) và nhỏ hơn một phần tử (có trong mảng) ⇒ không là min, không là max.

In [14]:
def find_first_three(nums: List[int]) -> int:
    if len(nums) < 3:
        return -1
    return sorted(nums[:3])[1]

print(find_first_three([3, 2, 1, 4]), find_first_three([1, 2]), find_first_three([2, 1, 3]))

2 -1 2


In [15]:
from typing import List

class SolutionFirstThree:
    def findNonMinOrMax(self, nums: List[int]) -> int:
        if len(nums) < 3:
            return -1
        return sorted(nums[:3])[1]

print(SolutionFirstThree().findNonMinOrMax([5, 9, 1, 7]))        # trung vị của 5, 9, 1 là 5

5


**Độ phức tạp:** chỉ đọc 3 phần tử, sắp xếp 3 số ⇒ Time `O(1)`, Space `O(1)` (lát cắt 3 phần tử). Đây là cách tối ưu nhất.

## 8. TRACE từng bước

Trace Cách 2 (`min`/`max` rồi quét) với `nums = [3, 2, 1, 4]`:

| bước | `x` | `lo = 1`, `hi = 4`: `lo < x < hi`? | hành động |
|---|---|---|---|
| 1 | 3 | `1 < 3 < 4` đúng | trả `3` |

(Nếu đi tiếp, `x = 2` cũng hợp lệ, nhưng ta chỉ cần **một** đáp án và trả ngay.)

Trace Cách 4 (3 phần tử đầu) với `[3, 2, 1, 4]`: ba phần tử đầu `3, 2, 1`, sắp xếp `[1, 2, 3]`, phần tử giữa `2` ⇒ trả `2` (khớp ví dụ 1 của đề). Với `[1, 2]`: độ dài `2 < 3` ⇒ `-1`.

In [16]:
def trace_min_max(nums):
    print("nums =", nums)
    if len(nums) < 3:
        print("  độ dài < 3 -> -1")
        return -1
    lo, hi = min(nums), max(nums)
    print(f"  lo={lo}, hi={hi}")
    for x in nums:
        ok = lo < x < hi
        print(f"  x={x}: {lo} < {x} < {hi}? {ok}")
        if ok:
            print("  -> trả", x)
            return x
    return -1

trace_min_max([3, 2, 1, 4])
trace_min_max([1, 2])
trace_min_max([1, 5, 9])

nums = [3, 2, 1, 4]
  lo=1, hi=4
  x=3: 1 < 3 < 4? True
  -> trả 3
nums = [1, 2]
  độ dài < 3 -> -1
nums = [1, 5, 9]
  lo=1, hi=9
  x=1: 1 < 1 < 9? False
  x=5: 1 < 5 < 9? True
  -> trả 5


5

In [17]:
def trace_first_three(nums):
    print("nums =", nums)
    if len(nums) < 3:
        print("  -> -1")
        return -1
    three = nums[:3]
    srt = sorted(three)
    print(f"  ba phần tử đầu {three}, sắp xếp {srt}, phần tử giữa {srt[1]}")
    return srt[1]

trace_first_three([3, 2, 1, 4])
trace_first_three([9, 1, 5, 2, 7])

nums = [3, 2, 1, 4]
  ba phần tử đầu [3, 2, 1], sắp xếp [1, 2, 3], phần tử giữa 2
nums = [9, 1, 5, 2, 7]
  ba phần tử đầu [9, 1, 5], sắp xếp [1, 5, 9], phần tử giữa 5


5

## 9. Code nộp LeetCode

Dùng Cách 4 (`O(1)`) hoặc Cách 2 (`min`/`max`). Ta nộp Cách 4 nhưng kèm kiểm tra `n < 3`. Đúng class `Solution` và method `findNonMinOrMax`.

In [18]:
from typing import List

class Solution:
    def findNonMinOrMax(self, nums: List[int]) -> int:
        if len(nums) < 3:
            return -1
        return sorted(nums[:3])[1]

In [19]:
print(Solution().findNonMinOrMax([3, 2, 1, 4]))   # 2  (3 cũng hợp lệ)
print(Solution().findNonMinOrMax([1, 2]))         # -1
print(Solution().findNonMinOrMax([2, 1, 3]))      # 2

2
-1
2


## 10. Vì sao thuật toán đúng

**Trường hợp `n < 3`.** Các giá trị đôi một khác nhau nên mảng có đúng `n` giá trị phân biệt. Nếu `n = 1` thì phần tử duy nhất vừa là min vừa là max. Nếu `n = 2` thì một phần tử là min, phần tử còn lại là max. Không còn phần tử nào ⇒ đáp án `-1`. Ta trả `-1` đúng.

**Trường hợp `n ≥ 3`: luôn tồn tại đáp án.** Min và max là **hai phần tử khác nhau** (do `n ≥ 2` và khác nhau từng đôi), nên còn ít nhất `n - 2 ≥ 1` phần tử khác. Mỗi phần tử đó **lớn hơn min và nhỏ hơn max** (vì khác nhau, nên nghiêm ngặt) ⇒ hợp lệ.

**Cách 2 đúng.** Điều kiện `lo < x < hi` chính là "không phải min, không phải max". Quét toàn mảng chắc chắn gặp một phần tử như vậy.

**Cách 3 đúng.** Sau sắp xếp `a_0 < a_1 < ... < a_{n-1}` (nghiêm ngặt). `a_0` là min, `a_{n-1}` là max, `a_1` với `1 ≤ n - 2` khi `n ≥ 3` nên `a_0 < a_1 < a_{n-1}` ⇒ hợp lệ.

**Cách 4 đúng (thủ thuật 3 phần tử đầu).** Gọi ba phần tử đầu là `p, q, r` (đôi một khác nhau). Gọi `m` là trung vị của chúng và giả sử sau khi sắp xếp ba số là `x < m < z` (với `{x, m, z} = {p, q, r}`). Cả `x` và `z` **đều có trong mảng**. Vậy trong mảng có phần tử **nhỏ hơn `m`** (`x`) ⇒ `m` không phải min; có phần tử **lớn hơn `m`** (`z`) ⇒ `m` không phải max. Điều này không cần biết min/max thực sự của mảng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (tối đa 100).

| Cách | Time | Space |
|---|---|---|
| 1. thử từng phần tử | `O(n²)` | `O(1)` |
| 2. `min`/`max` rồi quét | `O(n)` | `O(1)` |
| 3. sắp xếp, lấy `a[1]` | `O(n log n)` | `O(n)` |
| 4. trung vị 3 phần tử đầu | `O(1)` | `O(1)` |

Cách 4 không đọc hết mảng nên là `O(1)` — đây là ví dụ cho thấy **tính chất của dữ liệu** (khác nhau từng đôi) giúp loại bỏ cả vòng lặp. Cách 2 tối ưu nếu bạn muốn đáp án "tự nhiên" (không dùng thủ thuật).

## 12. Bộ test

### 12.1. Ví dụ của đề

Vì đề cho phép **mọi đáp án hợp lệ**, ta dùng `is_valid` (đã viết ở mục 3.5) thay vì so sánh `==`; chỉ ví dụ có đáp án **duy nhất** mới so `==`.

In [20]:
sols = [SolutionMinMax(), SolutionSorted(), SolutionFirstThree(), Solution()]
funcs = [find_brute, find_min_max, find_sorted, find_first_three] + [s.findNonMinOrMax for s in sols]

for f in funcs:
    assert is_valid([3, 2, 1, 4], f([3, 2, 1, 4]))
    assert f([3, 2, 1, 4]) in (2, 3)
    assert f([1, 2]) == -1
    assert f([2, 1, 3]) == 2            # đáp án duy nhất
print("Ví dụ của đề: tất cả cách đều đúng")

Ví dụ của đề: tất cả cách đều đúng


### 12.2. Edge case (chỉ dùng input hợp lệ: số dương, đôi một khác nhau)

- `n = 1`: `[5]` → `-1`.
- `n = 2`: `[1, 2]`, `[2, 1]`, `[100, 1]` → `-1`.
- `n = 3`: mọi hoán vị của `{1, 2, 3}` → `2` (duy nhất).
- Min và max đứng đầu / cuối: `[1, 5, 9]`, `[9, 5, 1]`, `[1, 9, 5]`.
- Phần tử giữa không ở chỉ số 1: `[1, 3, 9, 5]` ⇒ ba phần tử đầu `1, 3, 9` ⇒ trung vị `3`.
- Chuỗi dài 100: `list(range(1, 101))`, `list(range(100, 0, -1))`.

In [21]:
for f in funcs:
    assert f([5]) == -1
    for nums in ([1, 2], [2, 1], [100, 1]):
        assert f(nums) == -1
    import itertools
    for perm in itertools.permutations([1, 2, 3]):
        assert f(list(perm)) == 2
    for nums in ([1, 5, 9], [9, 5, 1], [1, 9, 5], [1, 3, 9, 5],
                 list(range(1, 101)), list(range(100, 0, -1))):
        assert is_valid(nums, f(nums)), (f, nums[:6])
print("Edge case OK")

Edge case OK


### 12.3. Phản ví dụ cho cách làm sai

Các cách sai hay gặp (chi tiết ở mục 13): luôn trả `nums[1]`, và tự tạo số `max - 1`. Ta xác nhận bằng `is_valid`.

In [22]:
def sai_nums_1(nums):
    return nums[1] if len(nums) >= 3 else -1

def sai_max_tru_1(nums):
    return max(nums) - 1 if len(nums) >= 3 else -1

print("sai_nums_1([2,1,3]) =", sai_nums_1([2, 1, 3]), "| hợp lệ?", is_valid([2, 1, 3], sai_nums_1([2, 1, 3])))
print("sai_max_tru_1([1,5,3]) =", sai_max_tru_1([1, 5, 3]), "| hợp lệ?", is_valid([1, 5, 3], sai_max_tru_1([1, 5, 3])))
assert not is_valid([2, 1, 3], sai_nums_1([2, 1, 3]))
assert not is_valid([1, 5, 3], sai_max_tru_1([1, 5, 3]))

sai_nums_1([2,1,3]) = 1 | hợp lệ? False
sai_max_tru_1([1,5,3]) = 4 | hợp lệ? False


### 12.4. Đối chiếu ngẫu nhiên (có seed) + vét cạn mảng nhỏ

Sinh mảng **hợp lệ theo constraints**: độ dài `n` từ `1` đến `100`, giá trị lấy bằng `random.sample` từ `1..100` (không lặp). Với mỗi mảng, **mọi cách** phải trả đáp án hợp lệ theo `is_valid`; ngoài ra `-1` đúng khi `n < 3`. Thêm vét cạn: với `n ≤ 5` và giá trị trong `1..6`, thử **mọi** mảng (mọi hoán vị mọi tập con).

In [23]:
import random, itertools

random.seed(2733)
for _ in range(3000):
    n = random.randint(1, 100)
    nums = random.sample(range(1, 101), n)
    assert len(set(nums)) == n and all(1 <= x <= 100 for x in nums)
    for f in funcs:
        ans = f(nums[:])
        assert is_valid(nums, ans), (nums[:8], f, ans)
    assert (find_min_max(nums) == -1) == (n < 3)
print("OK: 3000 mảng ngẫu nhiên; mọi cách trả đáp án hợp lệ")

# vét cạn: mọi mảng đôi một khác nhau với n <= 5 và giá trị trong 1..6
total = 0
for n in range(1, 6):
    for nums in itertools.permutations(range(1, 7), n):
        for f in funcs:
            assert is_valid(list(nums), f(list(nums))), (nums, f)
        total += 1
print("OK: vét cạn", total, "mảng nhỏ")

OK: 3000 mảng ngẫu nhiên; mọi cách trả đáp án hợp lệ


OK: vét cạn 1236 mảng nhỏ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Luôn trả `nums[1]`

In [24]:
from typing import List

def sai_tra_nums_1(nums: List[int]) -> int:
    return nums[1] if len(nums) >= 3 else -1

print("sai :", sai_tra_nums_1([2, 1, 3]))                       # 1: đó là min!
print("đúng:", Solution().findNonMinOrMax([2, 1, 3]))           # 2
assert sai_tra_nums_1([2, 1, 3]) == 1 and min([2, 1, 3]) == 1

sai : 1
đúng: 2


**Vì sao sai:** "phần tử thứ hai trong mảng" không có liên hệ gì với "phần tử ở giữa theo giá trị". `nums[1] = 1` là phần tử **nhỏ nhất** của `[2, 1, 3]`. Phải so sánh **giá trị**, không dựa vào **vị trí**. (Thủ thuật 3 phần tử đầu hợp lệ vì ta **sắp xếp** 3 phần tử rồi lấy giữa theo giá trị.)

### Lỗi 2: Quên trường hợp `n < 3`

In [25]:
def sai_quen_n_nho(nums: List[int]) -> int:
    return sorted(nums)[1]

for nums in ([5], [1, 2]):
    try:
        print("sai", nums, "->", sai_quen_n_nho(nums))
    except IndexError as e:
        print("sai", nums, "-> IndexError:", e)
print("đúng:", Solution().findNonMinOrMax([5]), Solution().findNonMinOrMax([1, 2]))
assert sai_quen_n_nho([1, 2]) == 2           # trả max, sai (đúng là -1)

sai [5] -> IndexError: list index out of range
sai [1, 2] -> 2
đúng: -1 -1


**Vì sao sai:** với `n = 1` truy cập chỉ số `1` gây `IndexError`; với `n = 2`, `sorted(nums)[1]` là **max** (sai, đáng lẽ `-1`). Luôn xử lý biên `n < 3` **trước**.

### Lỗi 3: Trả về chỉ số thay vì giá trị

In [26]:
def sai_tra_chi_so(nums: List[int]) -> int:
    if len(nums) < 3:
        return -1
    lo, hi = min(nums), max(nums)
    for i, x in enumerate(nums):
        if lo < x < hi:
            return i                         # SAI: đề cần GIÁ TRỊ
    return -1

print("sai :", sai_tra_chi_so([10, 30, 20]))        # 2 (chỉ số), nhưng 2 không thuộc nums
print("đúng:", Solution().findNonMinOrMax([10, 30, 20]))   # 20
nums = [10, 30, 20]
print(is_valid(nums, sai_tra_chi_so(nums)), is_valid(nums, Solution().findNonMinOrMax(nums)))
assert not is_valid(nums, sai_tra_chi_so(nums))

sai : 2
đúng: 20
False True


**Vì sao sai:** đề yêu cầu trả về **số** (phần tử) trong mảng, không phải vị trí của nó. (Khi `nums = [10, 30, 20]` đáp án hợp lệ là `20`; hãy kiểm tra: `min = 10`, `max = 30`, nên `20` là đáp án duy nhất.)

### Lỗi 4: Tự "chế" số không nằm trong mảng

In [27]:
def sai_che_so(nums: List[int]) -> int:
    if len(nums) < 3:
        return -1
    return max(nums) - 1                      # SAI: max - 1 có thể không có trong mảng

print("sai :", sai_che_so([1, 5, 3]), "(4 không có trong mảng)")
print("đúng:", Solution().findNonMinOrMax([1, 5, 3]))
assert not is_valid([1, 5, 3], sai_che_so([1, 5, 3]))

sai : 4 (4 không có trong mảng)
đúng: 3


**Vì sao sai:** đáp án phải là **một phần tử của `nums`**. `max - 1` chỉ là một số bất kỳ gần max; nó có thể không nằm trong mảng (như `4` với `[1, 5, 3]`) hoặc rơi đúng vào min. Luôn trả một phần tử **lấy từ mảng**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "trả về **bất kỳ** phần tử thoả điều kiện" ⇒ tìm cách **chắc chắn có đáp án** rồi chọn lấy cái dễ nhất; test bằng **checker** `is_valid`, không bằng `==`.
- Đọc kỹ ràng buộc **đặc biệt** ("các phần tử khác nhau") — đó thường là chìa khoá để đơn giản hoá (ở đây: chỉ cần `n ≥ 3`).
- **Trung vị của 3 phần tử** là thủ thuật nhỏ rất hay: cho phép bỏ qua phần còn lại của mảng.
- "Không phải min, không phải max" ⇒ nghĩ `lo < x < hi` hoặc "bỏ min và max, lấy phần tử còn lại".
- Chỉ số mảng ≠ giá trị mảng: cẩn thận khi trả kết quả.
- `next((x for x in nums if điều_kiện), default)` là mẫu "tìm phần tử đầu tiên thoả điều kiện".

## 15. Biến thể của bài

1. **Cho phép giá trị trùng nhau**: lúc đó điều kiện `n ≥ 3` không còn đủ (ví dụ `[2, 2, 2]`, `[1, 1, 2]` không có đáp án). Dùng `min`/`max` rồi quét `lo < x < hi` — cách 2 vẫn đúng, còn cách 3, 4 thì **phải kiểm tra lại**.
2. **Đếm** số phần tử không phải min/max: `n - nums.count(lo) - nums.count(hi)` (nếu `lo != hi`).
3. **Phần tử lớn thứ hai** / **thứ ba** (LeetCode 414 Third Maximum Number).
4. **Phần tử lớn thứ `k`** (LeetCode 215 Kth Largest Element): `heapq` hoặc quickselect.
5. **Trung vị** của mảng (LeetCode 4 Median of Two Sorted Arrays là bài khó).
6. **Trả về tất cả** phần tử không phải min/max, theo thứ tự tăng dần: `sorted(nums)[1:-1]`.

In [28]:
def count_middle(nums):
    lo, hi = min(nums), max(nums)
    return len([x for x in nums if lo < x < hi])

def all_middle_sorted(nums):
    return sorted(nums)[1:-1] if len(nums) >= 3 else []

def find_with_duplicates(nums):
    lo, hi = min(nums), max(nums)
    return next((x for x in nums if lo < x < hi), -1)

print(count_middle([3, 2, 1, 4]), all_middle_sorted([3, 2, 1, 4]))      # 2 [2, 3]
print(find_with_duplicates([2, 2, 2]), find_with_duplicates([1, 2, 2, 3]))   # -1 2

2 [2, 3]
-1 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. thử từng phần tử | với mỗi `x` kiểm tra có nhỏ hơn/lớn hơn | `O(n²)` | `O(1)` | hiểu đề, oracle |
| 2. `min`/`max` rồi quét | `lo < x < hi` | `O(n)` | `O(1)` | đúng cả khi có trùng giá trị (với kiểm tra) |
| 3. sắp xếp, lấy `a[1]` | phần tử thứ hai nhỏ nhất | `O(n log n)` | `O(n)` | code ngắn, `n` nhỏ |
| 4. trung vị 3 phần tử đầu | không cần đọc cả mảng | `O(1)` | `O(1)` | **cách nộp bài khuyến nghị** (khi giá trị khác nhau) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `1 < 5 < 9`, `sorted([5, 9, 1])[1]`, `next((x for x in [2, 4, 6] if x > 3), -1)`, `[7, 8][:3]`.

**Bài 2 (tính tay).** `nums = [10, 4, 7, 1]`: min, max là gì, những phần tử nào hợp lệ? Cách 4 trả về gì?

**Bài 3 (điền TODO).** Hoàn thành `find_todo(nums)` bằng `min`/`max`.

In [29]:
def find_todo(nums):
    # TODO 1: nếu len(nums) < 3 thì return -1
    # TODO 2: lo, hi = min(nums), max(nums)
    # TODO 3: duyệt nums, trả phần tử đầu tiên có lo < x < hi
    # TODO 4: return -1 ở cuối
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `find_first_three2(nums)` bằng trung vị của 3 phần tử đầu mà không nhìn Cách 4. Kiểm tra với ba ví dụ của đề (nhớ dùng `is_valid` cho ví dụ có nhiều đáp án).

**Bài 5 (biến thể / mini challenge).** (a) Viết `third_max(nums)` (LeetCode 414): trả về **số lớn thứ ba** khác nhau trong mảng, nếu không có thì trả số lớn nhất. (b) Viết `find_non_min_max_dups(nums)` cho mảng **có thể trùng giá trị**: trả một phần tử **khác min và khác max** hoặc `-1`. Rồi **tự viết test** chứng minh cách "3 phần tử đầu" sai khi có trùng (tìm một phản ví dụ).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True`; `5`; `4`; `[7, 8]`. Bài 2: min `1`, max `10`; các số `4`, `7` hợp lệ.
- **Mức 2 (vừa):** Bài 3: bốn TODO chính là Cách 2. Bài 4: `if len(nums) < 3: return -1` rồi `sorted(nums[:3])[1]`.
- **Mức 3 (sâu):** Bài 5(a): lấy `sorted(set(nums), reverse=True)`; nếu có ≥ 3 phần tử trả phần tử chỉ số `2`, ngược lại trả phần tử chỉ số `0`. Bài 5(b): giống Cách 2 (`lo < x < hi`), dùng `next(..., -1)`; phản ví dụ cho 3 phần tử đầu: `[1, 1, 1, 2, 3]` — ba phần tử đầu đều bằng `1` (trung vị `1` chính là min) nhưng đáp án thật là `2`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
assert (1 < 5 < 9) is True
assert sorted([5, 9, 1])[1] == 5
assert next((x for x in [2, 4, 6] if x > 3), -1) == 4
assert [7, 8][:3] == [7, 8]

# Bài 2: [10, 4, 7, 1]: min 1, max 10, hợp lệ {4, 7}; cách 4: 3 phần tử đầu 10, 4, 7 -> sorted [4, 7, 10] -> 7
nums = [10, 4, 7, 1]
assert (min(nums), max(nums)) == (1, 10)
assert [x for x in nums if min(nums) < x < max(nums)] == [4, 7]
assert Solution().findNonMinOrMax(nums) == 7 and is_valid(nums, 7)

# Bài 3
def find_todo_ans(nums):
    if len(nums) < 3:
        return -1
    lo, hi = min(nums), max(nums)
    for x in nums:
        if lo < x < hi:
            return x
    return -1

assert is_valid([3, 2, 1, 4], find_todo_ans([3, 2, 1, 4]))
assert find_todo_ans([1, 2]) == -1
assert find_todo_ans([2, 1, 3]) == 2

# Bài 4
def find_first_three2(nums):
    if len(nums) < 3:
        return -1
    return sorted(nums[:3])[1]

assert is_valid([3, 2, 1, 4], find_first_three2([3, 2, 1, 4]))
assert find_first_three2([1, 2]) == -1
assert find_first_three2([2, 1, 3]) == 2

# Bài 5
def third_max(nums):
    uniq = sorted(set(nums), reverse=True)
    return uniq[2] if len(uniq) >= 3 else uniq[0]

assert third_max([3, 2, 1]) == 1
assert third_max([1, 2]) == 2
assert third_max([2, 2, 3, 1]) == 1

def find_non_min_max_dups(nums):
    lo, hi = min(nums), max(nums)
    return next((x for x in nums if lo < x < hi), -1)

assert find_non_min_max_dups([1, 1, 1, 2, 3]) == 2
assert find_non_min_max_dups([2, 2, 2]) == -1
# phản ví dụ cho "3 phần tử đầu" khi có trùng:
assert not is_valid([1, 1, 1, 2, 3], find_first_three2([1, 1, 1, 2, 3]))      # trả 1 = min
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (giá trị khác nhau ⇒ trung vị của 3 phần tử đầu)

```python
from typing import List

class Solution:
    def findNonMinOrMax(self, nums: List[int]) -> int:
        if len(nums) < 3:
            return -1
        return sorted(nums[:3])[1]
```

**Checklist**

- [ ] Xử lý `n < 3` ⇒ `-1` **trước** mọi thứ?
- [ ] Trả về **giá trị** (phần tử của `nums`), không phải chỉ số, không phải số tự chế?
- [ ] Nếu dùng thủ thuật 3 phần tử đầu: có **sắp xếp** ba số rồi lấy giữa theo **giá trị** (không phải `nums[1]`)?
- [ ] Nhớ rằng thủ thuật dựa vào "các giá trị khác nhau từng đôi"; nếu có trùng phải dùng cách `min`/`max`?
- [ ] Test với checker `is_valid` (vì có nhiều đáp án đúng), không dùng `==` bừa bãi?
- [ ] Đã thử `[5]`, `[1, 2]`, hoán vị của `{1,2,3}`, `[2,1,3]`, mảng dài 100?
- [ ] Signature đúng `findNonMinOrMax(self, nums: List[int]) -> int` và có `from typing import List`?